# Multiclass Intrusion Detection on NSL-KDD (normal / DoS / Probe / R2L / U2R)

> **Research notebook, preserved from the original project (Mohammed Ali Cheddad).** The production pipeline used by the IDS application lives in `app/`; the 1D-CNN in `app/models/cnn.py` is refactored from the architecture below. The application does not depend on running these cells.

## Objective
Classify each connection into one of five categories, after consolidating the raw NSL-KDD attack labels into DoS, Probe, R2L and U2R.

## Preprocessing
Label consolidation, per-column standardisation fitted on the training split and replayed on the test split, one-hot encoding of the categorical features, and a single label encoder shared by train and test.

## Model
A Conv1D network (two convolution and pooling blocks followed by dense layers) and a Random Forest baseline.

## Evaluation
Train on KDDTrain+ and evaluate on the official **KDDTest+** split: accuracy, per-class precision/recall/F1, ROC/AUC and confusion matrix.

## Results
KDDTest+ accuracy is around 75%. Recall is very low for **R2L** and **U2R**, which have only 995 and 52 training records. This imbalance is the motivation for the GAN augmentation experiment in `app/gan/`; measured before/after numbers are in `docs/RESULTS.md`.


## Imports

In [ ]:
# Core data-handling libraries
import numpy as np
import pandas as pd

# Preprocessing utilities
from sklearn import preprocessing
from sklearn.preprocessing import LabelBinarizer, LabelEncoder, StandardScaler

# Visualisation
import itertools
from itertools import cycle
import matplotlib.pyplot as plt
import seaborn as sns

# Evaluation metrics
from sklearn import metrics
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (auc, classification_report, confusion_matrix,
                              roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split

# Deep learning
import tensorflow as tf
from tensorflow.keras.layers import Conv1D, Dense, Dropout, Flatten, Input, MaxPool1D
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical

## Load the Dataset

In [ ]:
features=["duration","protocol_type","service","flag","src_bytes","dst_bytes","land","wrong_fragment","urgent","hot",
          "num_failed_logins","logged_in","num_compromised","root_shell","su_attempted","num_root","num_file_creations","num_shells",
          "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count","srv_count","serror_rate","srv_serror_rate",
          "rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count", 
          "dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
          "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate","label","difficulty"]

In [ ]:
train='nsl-kdd/KDDTrain+.txt'
test='nsl-kdd/KDDTest+.txt'
test21='nsl-kdd/KDDTest-21.txt'
train_data=pd.read_csv(train,names=features)
train_data

In [ ]:
train_data.drop(['difficulty'],axis=1,inplace=True)


In [ ]:
test_data = pd.read_csv(test,names=features)
test_data.drop(['difficulty'],axis=1,inplace=True)

In [ ]:
train_data['label'].value_counts()


## Label Consolidation

Map the 22 raw NSL-KDD attack labels down to 4 broad categories (Dos, R2L, Probe, U2R) plus 'normal'.

In [ ]:
def change_label(df):
    """Consolidate the raw NSL-KDD attack labels into 4 broad attack
    categories (Dos, R2L, Probe, U2R); 'normal' rows are left unchanged."""
    mapping = {}
    mapping.update({k: "Dos" for k in [
        "apache2", "back", "land", "neptune", "mailbomb", "pod",
        "processtable", "smurf", "teardrop", "udpstorm", "worm"]})
    mapping.update({k: "R2L" for k in [
        "ftp_write", "guess_passwd", "httptunnel", "imap", "multihop",
        "named", "phf", "sendmail", "snmpgetattack", "snmpguess", "spy",
        "warezclient", "warezmaster", "xlock", "xsnoop"]})
    mapping.update({k: "Probe" for k in [
        "ipsweep", "mscan", "nmap", "portsweep", "saint", "satan"]})
    mapping.update({k: "U2R" for k in [
        "buffer_overflow", "loadmodule", "perl", "ps", "rootkit",
        "sqlattack", "xterm"]})
    df["label"] = df["label"].replace(mapping)

In [ ]:
change_label(train_data)

In [ ]:
change_label(test_data)

In [ ]:
test_data['label'].value_counts()


In [ ]:
train_data['label'].value_counts()


## Exploratory Data Analysis

In [ ]:
def pie_plot(df, cols_list, rows, cols):
    fig, axes = plt.subplots(rows, cols)
    for ax, col in zip(axes.ravel(), cols_list):
        df[col].value_counts().plot(ax=ax, kind='pie', figsize=(15, 15), fontsize=10, autopct='%1.0f%%')
        ax.set_title(str(col), fontsize = 12)
    plt.show()

In [ ]:
pie_plot(train_data, ['protocol_type', 'label'], 1, 2)

## Preprocessing

In [ ]:
multi_data = train_data.copy()
multi_label = pd.DataFrame(multi_data.label)

In [ ]:
multi_data_test = test_data.copy()
multi_label_test = pd.DataFrame(multi_data_test.label)

In [ ]:
multi_label

## Feature Standardization

Numeric columns are standardized using scalers fit on the **training data only**, then reused on the test data (see notes below).

In [ ]:
def fit_standardization(df, columns):
    """Fit one StandardScaler per numeric column on the TRAINING data and
    apply it in place. Returns the fitted scalers so the same
    transformation can be replayed on the test data."""
    scalers = {}
    for col in columns:
        scaler = StandardScaler()
        df[col] = scaler.fit_transform(df[[col]])
        scalers[col] = scaler
    return df, scalers


def apply_standardization(df, scalers):
    """Transform columns using scalers already fitted on the training data."""
    for col, scaler in scalers.items():
        if col in df.columns:
            df[col] = scaler.transform(df[[col]])
    return df


numeric_col = multi_data.select_dtypes(include="number").columns
multi_data, fitted_scalers = fit_standardization(multi_data, numeric_col)

In [ ]:
# Reuse the scalers fitted on the training data above -- do NOT re-fit on
# the test set (see the note in the previous cell).
numeric_col_test = multi_data_test.select_dtypes(include="number").columns
multi_data_test = apply_standardization(multi_data_test, fitted_scalers)

In [ ]:
multi_data_test  # display the standardized test set

## Label Encoding and One-Hot Encoding

In [ ]:
# Label-encode the multi-class labels (Dos, normal, Probe, R2L, U2R) as
# integers 0-4. `le2` is fit ONCE here, on the training labels, and reused
# (via .transform, not .fit_transform) on the test labels in the next cell
# so the integer codes line up between train and test.
le2 = preprocessing.LabelEncoder()
enc_label = multi_label.apply(le2.fit_transform)
multi_data["intrusion"] = enc_label
multi_data

In [ ]:
enc_label_test = multi_label_test.apply(le2.transform)
multi_data_test["intrusion"] = enc_label_test
multi_data_test

In [ ]:
multi_data.drop(labels= [ 'label'], axis=1, inplace=True)
multi_data

In [ ]:
multi_data_test.drop(labels= [ 'label'], axis=1, inplace=True)


In [ ]:
np.array(test_data["label"])

In [ ]:
test_data["label"].unique

In [ ]:
# one-hot-encoding attack label
multi_data = pd.get_dummies(multi_data,columns=['protocol_type','service','flag'],prefix="",prefix_sep="")  
multi_data

In [ ]:
# one-hot-encoding attack label
multi_data_test = pd.get_dummies(multi_data_test,columns=['protocol_type','service','flag'],prefix="",prefix_sep="")  
multi_data_test

In [ ]:
unique_columns_multi_data = set(multi_data.columns) - set(multi_data_test.columns)
unique_columns_multi_data_test = set(multi_data_test.columns) - set(multi_data.columns)

print("Columns in multi_data but not in multi_data_test:", unique_columns_multi_data)
print("Columns in multi_data_test but not in multi_data:", unique_columns_multi_data_test)

# BUG FIX: the original code only dropped the train-only columns from the
# training set, and never reconciled the test-only columns -- leaving
# train and test with a different number of features, which crashes the
# model at prediction time (shape mismatch). Align both frames to the
# same column set; a category seen only at test time carries no learned
# weights anyway, so dropping it is safe.
multi_data = multi_data.drop(columns=unique_columns_multi_data)
multi_data_test = multi_data_test.drop(columns=unique_columns_multi_data_test)
multi_data

## Train/Test Split for the Neural Network

In [ ]:
y_train_multi= multi_data[['intrusion']]
X_train_multi= multi_data.drop(labels=['intrusion'], axis=1)

print('X_train has shape:',X_train_multi.shape,'\ny_train has shape:',y_train_multi.shape)

In [ ]:
y_test_multi= multi_data_test[['intrusion']]
X_test_multi= multi_data_test.drop(labels=['intrusion'], axis=1)

print('X_train has shape:',X_test_multi.shape,'\ny_train has shape:',y_test_multi.shape)

In [ ]:
# Fit ONE LabelBinarizer on the training intrusion codes, and keep its
# one-hot output under a distinct name (`_ohe`) rather than overwriting
# `y_train_multi` -- the original notebook reused that name for the
# one-hot array, which later broke the Random Forest baseline below
# (RandomForestClassifier().fit(..., y_train_multi) silently trained on
# one-hot targets, and le2.inverse_transform() then failed on the
# resulting one-hot predictions).
label_binarizer = LabelBinarizer().fit(y_train_multi)
y_train_multi_ohe = label_binarizer.transform(y_train_multi)
y_train_multi_ohe

In [ ]:
y_test_multi_ohe = label_binarizer.transform(y_test_multi)
y_test_multi_ohe

In [ ]:
X_train = np.array(X_train_multi)
y_train = np.array(y_train_multi_ohe)

In [ ]:
X_test = np.array(X_test_multi)
y_test = np.array(y_test_multi_ohe)

In [ ]:
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
X_train.shape

In [ ]:
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))
X_test.shape

## CNN Model: Build, Train, and Evaluate

In [ ]:
# Keras 3 expects an explicit Input layer rather than passing
# `input_shape=` to the first layer.
model = Sequential([
    Input(shape=(X_train.shape[1], 1)),
    Conv1D(32, 3, padding="same", activation="relu"),
    MaxPool1D(pool_size=4),
    Dropout(0.2),
    Conv1D(32, 3, padding="same", activation="relu"),
    MaxPool1D(pool_size=4),
    Dropout(0.2),
    Flatten(),
    Dense(units=50),
    Dense(units=5, activation="softmax"),  # 5 intrusion classes
])

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


In [ ]:
model.summary()


In [ ]:
# Train on a GPU automatically when one is available, otherwise fall back
# to CPU so the notebook still runs on machines without a GPU.
device = "/GPU:0" if tf.config.list_physical_devices("GPU") else "/CPU:0"
with tf.device(device):
    history = model.fit(X_train, y_train, epochs=100, batch_size=5000, validation_split=0.2)

In [ ]:
print(X_train.dtype)
print(X_test.dtype)


In [ ]:
X_train = X_train.astype('float64')
X_test = X_test.astype('float64')



In [ ]:
X_train.shape

In [ ]:
gpus = tf.config.list_physical_devices("GPU")
print("Num GPUs Available:", len(gpus))
print("TensorFlow is using the GPU" if gpus else "TensorFlow is not using the GPU")

In [ ]:
tf.config.list_physical_devices('GPU')

In [ ]:
# predicting target attribute on testing dataset
test_results = model.evaluate(X_test, y_test, verbose=1)
print(f'Test results - Loss: {test_results[0]} - Accuracy: {test_results[1]*100}%')

In [ ]:
# Plot of accuracy vs epoch for train and test dataset
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title("Plot of accuracy vs epoch for train and test dataset")
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.show()

In [ ]:
y_pred_test = model.predict(X_test, batch_size=500)

# For Plot curve
y_pred_evaluation_test = to_categorical(np.argmax(y_pred_test, axis=1), 5)

# For evaluation
y_pred_argmax_test=(np.argmax(y_pred_test, axis=1))

y_test_argmax=(np.argmax(y_test, axis=1))

In [ ]:
# Calculating Area under the curve
def AUC(actual_class, pred_class, average = "micro"):
    
    #Making a set of all the unique classes 
    unique_class = set(actual_class)
    roc_auc_dict = {}
    for per_class in unique_class:
        
        #Making a list of all the classes except the current class 
        other_class = [x for x in unique_class if x != per_class]

        #Making the current class with label 1 and all other classes as a label 0
        new_actual_class = [0 if x in other_class else 1 for x in actual_class]
        new_pred_class = [0 if x in other_class else 1 for x in pred_class]

        # Calculating the roc_auc_score
        roc_auc = roc_auc_score(new_actual_class, new_pred_class, average = average)
        roc_auc_dict[per_class] = roc_auc

    return roc_auc_dict

In [ ]:
def plot_roc_curve(y_test,y_pred, classes):
    n_classes=len(classes)
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test[:, i], y_pred[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    fpr["micro"], tpr["micro"], _ = roc_curve(y_test.ravel(), y_pred.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
    lw=2
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))

    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])

    mean_tpr /= n_classes

    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

# Plot all ROC curves
    plt.figure(figsize=(10, 10))
    plt.plot(fpr["micro"],
    tpr["micro"],
    label="micro-average ROC curve (area = {0:0.2f})".format(roc_auc["micro"]),
    color="deeppink",
    linestyle=":",
    linewidth=4,)

    plt.plot(fpr["macro"],
    tpr["macro"],
    label="macro-average ROC curve (area = {0:0.2f})".format(roc_auc["macro"]),
    color="navy",
    linestyle=":",
    linewidth=4,
)

    colors = cycle(["aqua", "darkorange", "cornflowerblue"])
    for i, color in zip(range(n_classes), colors):
        plt.plot(
           fpr[i],
           tpr[i],
           color=color,
           lw=lw,
        label="ROC curve of {0} (area = {1:0.2f})".format(classes[i], roc_auc[i]),
    )

    plt.plot([0, 1], [0, 1], "k--", lw=lw)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("AUC")
    plt.legend(loc="lower right")
    plt.show()

In [ ]:
# (imports consolidated into the first cell)

### Reporting Helpers: ROC/AUC and Confusion Matrix

In [ ]:
def plot_confusion_matrix(cm, classes, normalize=False,
                           title="Confusion matrix", cmap=plt.cm.Blues):
    """Print and plot a confusion matrix, optionally row-normalized."""
    if normalize:
        cm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print("Confusion matrix, without normalization")

    plt.imshow(cm, interpolation="nearest", cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = ".2f" if normalize else "d"
    thresh = cm.max() / 2.0
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                  horizontalalignment="center",
                  color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel("True label")
    plt.xlabel("Predicted label")

In [ ]:
# predicting target attribute on testing dataset
test_results = model.evaluate(X_test, y_test, verbose=1)
print(f'Test results - Loss: {test_results[0]} - Accuracy: {test_results[1]*100}%')

In [ ]:
print('AUC Score is on Test : ' , AUC(y_test_argmax, y_pred_argmax_test))


In [ ]:
classes = list(le2.classes_)

print("Classification Report on Data Test\n",
      classification_report(y_test_argmax, y_pred_argmax_test, target_names=classes))

In [ ]:
# Compute confusion matrix
cnf_matrix = confusion_matrix(y_test_argmax, y_pred_argmax_test)

plt.figure(figsize=(7, 7))
plot_confusion_matrix(cnf_matrix, classes=classes, title="Test Confusion matrix")
plt.show()

### Baseline: Random Forest (multi-class)

In [ ]:
# NOTE: `y_train_multi` here is still the *integer-coded* intrusion column
# from cell 19 (not the one-hot `y_train_multi_ohe` used for the CNN
# above), which is what RandomForestClassifier needs for standard
# multi-class classification.
rf = RandomForestClassifier().fit(X_train_multi, y_train_multi.values.ravel())

In [ ]:
y_pred_test = rf.predict(X_test_multi)

y_pred_test

In [ ]:
y_pred_labels = le2.inverse_transform(y_pred_test)
y_pred_labels


In [ ]:
np.array(test_data["label"])

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(np.array(test_data["label"]), y_pred_labels)


In [ ]:
# Reuse the encoder's class order (see the note in the classification
# report cell above -- this hardcoded list did not match le2.classes_).
classes = list(le2.classes_)

# Plot non-normalized confusion matrix
plt.figure()
plot_confusion_matrix(cm, classes=classes, title="Confusion matrix, without normalization")

# Plot normalized confusion matrix
plt.figure()
plot_confusion_matrix(cm, classes=classes, normalize=True, title="Normalized confusion matrix")

plt.show()